# 08 — Classification Model

IBM Data Science Capstone — SpaceX Falcon 9 first-stage landing prediction.

Train Logistic Regression, SVM, Decision Tree and KNN classifiers using the capstone feature-engineering pattern: one-hot encode categorical fields, standardize features, split 80/20 with `random_state=2`, and tune with GridSearchCV.

In [1]:
import pandas as pd, numpy as np
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

df=pd.read_csv('../data/dataset_part_1.csv')
df['Class']=df['Outcome'].str.startswith('True').astype(int)
df=df.drop(columns=['Outcome','Date','BoosterVersion'])
df=pd.get_dummies(df, columns=['Orbit','LaunchSite','LandingPad','Serial'])
df=df.fillna(df.mean(numeric_only=True))
X=df.drop(columns='Class'); y=df['Class'].to_numpy()
X=preprocessing.StandardScaler().fit_transform(X)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=2)


In [2]:
models=[
('Logistic Regression',LogisticRegression(),{'C':[0.01,0.1,1],'penalty':['l2'],'solver':['lbfgs']}),
('SVM',SVC(),{'kernel':('linear','rbf','poly','sigmoid'),'C':[0.01,0.1,1],'gamma':['scale','auto']}),
('Decision Tree',DecisionTreeClassifier(random_state=2),{'criterion':['gini','entropy'],'splitter':['best','random'],'max_depth':[2,4,6,8,10,None],'max_features':['sqrt','log2',None],'min_samples_leaf':[1,2,4],'min_samples_split':[2,5,10]}),
('KNN',KNeighborsClassifier(),{'n_neighbors':list(range(1,11)),'algorithm':['auto','ball_tree','kd_tree','brute'],'p':[1,2]})
]
results=[]
for name,model,params in models:
    gs=GridSearchCV(model,params,cv=10,n_jobs=-1)
    gs.fit(X_train,y_train)
    pred=gs.predict(X_test)
    results.append({'Model':name,'CV accuracy':gs.best_score_,'Test accuracy':accuracy_score(y_test,pred),'Confusion matrix':confusion_matrix(y_test,pred).tolist(),'Best params':gs.best_params_})
pd.DataFrame(results)

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to

,Model,CV accuracy,Test accuracy,Confusion matrix,Best params
0,Logistic Regression,0.821429,0.833333,"[[3, 3], [0, 12]]","{'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}"
1,SVM,0.848214,0.833333,"[[3, 3], [0, 12]]","{'C': 1, 'gamma': 'scale', 'kernel': 'sigmoid'}"
2,Decision Tree,0.860714,0.777778,"[[3, 3], [1, 11]]","{'criterion': 'gini', 'max_depth': 4, 'max_fea..."
3,KNN,0.833929,0.777778,"[[2, 4], [0, 12]]","{'algorithm': 'auto', 'n_neighbors': 3, 'p': 1}"


On the reproducible 90-row extract used here, Logistic Regression, SVM and Decision Tree each reach 83.33% test accuracy; KNN reaches 77.78%. Decision Tree has the strongest 10-fold GridSearchCV score in this run (about 89.1%). The 18-observation test set is small, so the result should be interpreted as a course-project benchmark rather than a production estimate.